# Japan-Paw: indexación por tandas con estado en Drive

Usa este cuaderno con la versión corregida del indexador. Primero procesa hasta **100 archivos o 15 minutos**, con dos trabajadores. No lanza el catálogo entero ni publica resultados. Ejecuta una sola sesión por directorio de estado.

Los fallos de red quedan separados de los archivos incompatibles. La comprobación de una muestra de piezas no equivale a verificar todos los bytes del video. Consulta `docs/COLAB.md` para interpretar estados y reanudar.


In [ ]:
# 1. Obtener el código corregido. Sube el ZIP entregado junto a este cuaderno.
# Si ya publicaste las correcciones en GitHub, puedes cambiar SOURCE_MODE a "git".
from pathlib import Path
import json, os, shutil, subprocess, zipfile, hashlib, signal, datetime

SOURCE_MODE = "git"  # "upload" o "git"
PROJECT = Path("/content/ExtenJap")
REPOSITORY = "https://github.com/JuanPerezC893/ExtenJap.git"

if SOURCE_MODE == "upload":
    from google.colab import files
    uploaded = files.upload()
    archives = [Path(name) for name in uploaded if name.lower().endswith(".zip")]
    if len(archives) != 1:
        raise RuntimeError("Sube exactamente un ZIP del proyecto corregido.")
    if PROJECT.exists():
        raise RuntimeError("La carpeta /content/ExtenJap ya existe. Reutiliza las siguientes celdas o cambia PROJECT a una carpeta nueva; no se sobrescribe automáticamente.")
    with zipfile.ZipFile(archives[0]) as archive:
        # El bundle debe contener package.json y raw-catalog.json en su raíz.
        names = archive.namelist()
        if "package.json" not in names or "raw-catalog.json" not in names:
            raise RuntimeError("El ZIP debe contener package.json y raw-catalog.json en la raíz.")
        root = PROJECT.resolve()
        for member in archive.infolist():
            target = (root / member.filename).resolve()
            if not target.is_relative_to(root):
                raise RuntimeError("El ZIP contiene una ruta fuera de la carpeta destino.")
        PROJECT.mkdir(parents=True)
        archive.extractall(PROJECT)
elif SOURCE_MODE == "git":
    if not PROJECT.exists():
        subprocess.run(["git", "clone", REPOSITORY, str(PROJECT)], check=True)
    else:
        subprocess.run(["git", "fetch", "origin", "main"], cwd=PROJECT, check=True)
        subprocess.run(["git", "reset", "--hard", "origin/main"], cwd=PROJECT, check=True)
else:
    raise ValueError("SOURCE_MODE debe ser upload o git.")

source = (PROJECT / "indexer.mjs").read_text()
if "state-dir" not in source or "max-minutes" not in source:
    raise RuntimeError("Esta copia del indexador no contiene las opciones de reanudación. Carga la versión corregida antes de continuar.")
node_version = subprocess.check_output(["node", "--version"], text=True).strip()
print("Node:", node_version)
if int(node_version.lstrip("v").split(".")[0]) < 20:
    raise RuntimeError("Este flujo necesita Node.js 20 o posterior. Actualiza Node antes de instalar dependencias.")
subprocess.run(["npm", "ci"], cwd=PROJECT, check=True)
print("Código preparado en", PROJECT)


In [ ]:
# 2. Montar Drive y conservar las asociaciones existentes, solo al iniciar un estado nuevo.
from google.colab import drive

drive.mount("/content/drive")
STATE = Path("/content/drive/MyDrive/JapanPaw-index")
CATALOG = PROJECT / ("raw-catalog.json" if (PROJECT / "raw-catalog.json").exists() else "dist/indexed-catalog.json")
STATE.mkdir(parents=True, exist_ok=True)

registry = STATE / "verified-matches.json"
job_state = STATE / "indexer-state.json"
if registry.exists() or job_state.exists():
    print("Estado existente: se conserva sin sustituir por el del repositorio.")
else:
    # Primero se copian los torrents. Instalar el registro es el último paso.
    source_torrents = PROJECT / "dist/torrents"
    target_torrents = STATE / "dist/torrents"
    target_torrents.mkdir(parents=True, exist_ok=True)
    for path in source_torrents.glob("*.torrent"):
        destination = target_torrents / path.name
        if not destination.exists():
            shutil.copy2(path, destination)
    source_registry = PROJECT / "verified-matches.json"
    if source_registry.exists():
        json.loads(source_registry.read_text())  # No instalar JSON incompleto.
        temporary = STATE / "verified-matches.bootstrap.tmp"
        shutil.copy2(source_registry, temporary)
        os.replace(temporary, registry)
    print("Directorio de estado inicializado.")

# Guardar la entrada por su huella permite saber qué catálogo usó cada tanda.
catalog_hash = hashlib.sha256(CATALOG.read_bytes()).hexdigest()
input_copy = STATE / "inputs" / ("catalog-" + catalog_hash + ".json")
input_copy.parent.mkdir(parents=True, exist_ok=True)
if not input_copy.exists():
    shutil.copy2(CATALOG, input_copy)
try:
    revision = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROJECT, text=True, stderr=subprocess.DEVNULL).strip()
except subprocess.CalledProcessError:
    revision = "bundle local; conservar el ZIP del código"
metadata = {"recordedAt": datetime.datetime.now(datetime.timezone.utc).isoformat(), "catalogSha256": catalog_hash, "codeRevision": revision, "node": node_version}
(STATE / "inputs" / ("run-input-" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S") + ".json")).write_text(json.dumps(metadata, indent=2))
print("Estado persistente:", STATE)
print("No ejecutes otro indexador simultáneo sobre esta misma carpeta.")


In [ ]:
# 3. Ejecutar una tanda. Puedes repetir esta celda para continuar.
CONCURRENCY = 2
LIMIT = 100
MAX_MINUTES = 15
SERIES = []  # Ejemplo: ["185874"] o ["Sayonara Lara"]
RETRY_PENDING = True  # Respeta las pausas de proveedores; no evita un bloqueo.

# Auto-sincronizar git para asegurar que siempre corra el último código de GitHub
if (PROJECT / ".git").exists():
    try:
        subprocess.run(["git", "fetch", "origin", "main"], cwd=PROJECT, check=True, stderr=subprocess.DEVNULL)
        subprocess.run(["git", "reset", "--hard", "origin/main"], cwd=PROJECT, check=True, stderr=subprocess.DEVNULL)
    except Exception as e:
        print("Aviso al auto-sincronizar git:", e)
try:
    current_rev = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=PROJECT, text=True).strip()
    print(f"Versión de código en {PROJECT}: {current_rev}")
except Exception:
    pass

command = ["node", "indexer.mjs", "--catalog", str(CATALOG), "--state-dir", str(STATE),
           "--concurrency", str(CONCURRENCY), "--limit", str(LIMIT), "--max-minutes", str(MAX_MINUTES)]
if SERIES:
    command.extend(["--series", *map(str, SERIES)])
if RETRY_PENDING:
    command.append("--retry-pending")

print("Comando:", " ".join(command))
process = subprocess.Popen(command, cwd=PROJECT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                           text=True, bufsize=1, start_new_session=True)
try:
    for line in process.stdout:
        print(line, end="")
    INDEXER_EXIT = process.wait()
except KeyboardInterrupt:
    print("\nInterrupción solicitada. Esperando el guardado del indexador...")
    if process.poll() is None:
        os.killpg(process.pid, signal.SIGINT)
    try:
        remaining, _ = process.communicate(timeout=45)
        if remaining:
            print(remaining, end="")
        INDEXER_EXIT = process.returncode
    except subprocess.TimeoutExpired:
        raise RuntimeError("El proceso sigue cerrando. No inicies otra tanda ni cierres el entorno; revisa el proceso antes de continuar.")

# Node puede informar 130 o ser terminado directamente por SIGINT (-2).
if INDEXER_EXIT == -signal.SIGINT:
    INDEXER_EXIT = 130
messages = {0: "Tanda terminada; revisar cuántos archivos se prepararon.",
            2: "Trabajo diferido por proveedores; revisar pausas antes de reintentar.",
            130: "Interrumpido; conservar y revisar el último punto de control."}
print("\nSalida:", INDEXER_EXIT, messages.get(INDEXER_EXIT, "Error fatal: revisar la salida antes de continuar."))
if INDEXER_EXIT not in (0, 2, 130):
    raise RuntimeError("El indexador terminó con un error fatal. No compilar automáticamente.")


In [ ]:
# 4. Compilar los avances guardados. No declara exitosos los archivos pendientes.
if INDEXER_EXIT not in (0, 2, 130):
    raise RuntimeError("Revisa el fallo del indexador antes de compilar.")
subprocess.run(["node", "build.mjs", "--state-dir", str(STATE)], cwd=PROJECT, check=True)
print("Publicación preparada en", STATE / "dist")


## Exportación opcional

Drive contiene el estado de trabajo. Esta última celda crea una copia adicional descargable de **todo el directorio de estado**, incluidos las pausas de proveedores, los registros y los torrents. No sube nada a GitHub. Conserva también el ZIP del código; al restaurar, usa una carpeta nueva y selecciona esa carpeta como `STATE`.

El montaje de Drive puede tardar en sincronizar. Una sesión eliminada de forma abrupta puede perder el trabajo posterior al último guardado disponible. Detén la tanda antes de exportar y no ejecutes otro escritor al mismo tiempo.


In [ ]:
# 5. Descargar un respaldo después de cerrar el indexador.
if "process" in globals() and process.poll() is None:
    raise RuntimeError("El indexador todavía está activo; termina la tanda antes de exportar.")
from google.colab import files
stamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
archive_base = Path("/content") / ("JapanPaw-estado-" + stamp)
archive_file = shutil.make_archive(str(archive_base), "zip", root_dir=STATE)
print("Respaldo:", archive_file)
files.download(archive_file)
